# Домашнее задание: Построение и анализ филогенетических деревьев

**Ген:** Цитохром c (CYCS)

## Установка зависимостей

In [ ]:
!pip install biopython -q
!apt-get install -y mafft -q

## Задание 1. Подготовка данных

In [ ]:
# Загрузка файла с локального компьютера
from google.colab import files
uploaded = files.upload()  # выбери sequence.fasta

In [ ]:
# Копируем и обрезаем до 100 последовательностей
import shutil, os
from Bio import SeqIO

shutil.copy(list(uploaded.keys())[0], "cycs_raw_full.fasta")

all_records = list(SeqIO.parse("cycs_raw_full.fasta", "fasta"))
print(f"Всего последовательностей: {len(all_records)}")

subset = all_records[:100]
SeqIO.write(subset, "cycs_raw.fasta", "fasta")
print(f"Оставлено для работы: {len(subset)}")
print(f"Размер файла: {os.path.getsize('cycs_raw.fasta')} байт")

In [ ]:
# 1.3 -- переименование заголовков
# Было:  >NP_061820.1 CYCS [organism=Homo sapiens] [GeneID=54205]
# Стало: >Homo_sapiens_NP_061820.1

import re

renamed_records = []

for rec in SeqIO.parse("cycs_raw.fasta", "fasta"):
    accession = rec.id

    match = re.search(r'\[organism=([^\]]+)\]', rec.description)
    if match:
        organism = match.group(1).replace(" ", "_")
    else:
        # Если тег organism отсутствует -- берём первые два слова описания
        words = rec.description.split()
        organism = "_".join(words[1:3]) if len(words) >= 3 else "Unknown"

    rec.id = f"{organism}_{accession}"
    rec.description = ""
    rec.name = ""
    renamed_records.append(rec)

SeqIO.write(renamed_records, "cycs_renamed.fasta", "fasta")
print(f"Переименовано: {len(renamed_records)} записей")

# Проверка первых 5 заголовков
print("\nПервые 5 заголовков:")
for rec in renamed_records[:5]:
    print(rec.id)

### Альтернативный вариант через awk (bash)

```bash
awk '/^>/ {
    acc = substr($1, 2)
    match($0, /\[organism=([^]]+)\]/, arr)
    org = arr[1]
    gsub(/ /, "_", org)
    print ">" org "_" acc
    next
} { print }' cycs_raw.fasta > cycs_renamed.fasta
```

## Задание 2. Множественное выравнивание (MAFFT)

In [ ]:
import subprocess

result = subprocess.run(
    ["mafft", "--auto", "--thread", "-1", "cycs_renamed.fasta"],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    with open("cycs_aligned.fasta", "w") as f:
        f.write(result.stdout)
    print("Выравнивание готово: cycs_aligned.fasta")
else:
    print("Ошибка MAFFT:")
    print(result.stderr[:500])

In [ ]:
# Проверка выравнивания
from Bio import AlignIO

alignment = AlignIO.read("cycs_aligned.fasta", "fasta")
total_chars = len(alignment) * alignment.get_alignment_length()
gap_count = sum(str(rec.seq).count('-') for rec in alignment)

print(f"Последовательностей: {len(alignment)}")
print(f"Длина выравнивания:   {alignment.get_alignment_length()} позиций")
print(f"Доля гэпов:           {gap_count / total_chars:.2%}")

In [ ]:
# Скачать готовое выравнивание на компьютер
from google.colab import files
files.download("cycs_aligned.fasta")